# Lesson 4: File Processing — CSV, JSON, Excel & Parquet

**Week 5 · Data Engineering Course**

---

You have data. Now: in what format does it land on disk?

So far this course has used CSV for everything, because CSV is where everyone starts. It is also the format that loses your types, mangles your dates, and takes ten times the space it needs. A data engineer chooses a format deliberately, for reasons they can explain.

This lesson teaches you:
- The four formats you will actually meet, and when each is the right answer
- CSV's traps: encoding, delimiters, quoting, and the fact that it has no types
- Flattening nested JSON with `json_normalize`, and why pipelines prefer JSON **Lines**
- Reading and writing Excel workbooks with several sheets
- **Parquet** — columnar, typed, compressed — and why warehouses are built on it
- A real size-and-speed comparison on the same dataset
- Partitioned datasets

**Install before starting:**
```
pip install pandas pyarrow openpyxl
```

`pyarrow` gives pandas Parquet support; `openpyxl` gives it `.xlsx` support.

In [ ]:
!pip install pandas pyarrow openpyxl

In [ ]:
import json
import time
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

DATA  = Path('data')
PAGES = DATA / 'pages'
CLEAN = DATA / 'clean'
OUT   = DATA / 'formats'          # everything this lesson writes goes here
for p in [CLEAN, OUT]:
    p.mkdir(parents=True, exist_ok=True)

print(f'pandas  {pd.__version__}')
import pyarrow
print(f'pyarrow {pyarrow.__version__}')

### The dataset for this lesson

We will use the job board from Lesson 1 — both saved pages this time, scraped offline so this notebook needs no network. Ten postings with a realistic mix of types: strings, an integer id, a date, a boolean, a float, and a missing value.

That mix is the point. A format that handles strings is easy; it is dates, booleans, nulls and numbers that separate the formats from each other.

In [ ]:
def parse_jobs(path):
    '''Scrape one saved job-board page into rows (the Lesson 1 parser).'''
    soup = BeautifulSoup(Path(path).read_text(encoding='utf-8'), 'html.parser')
    rows = []
    for card in soup.select('article.job-card'):
        def text_at(sel):
            found = card.select_one(sel)
            return found.get_text(strip=True) if found else None

        salary = text_at('.salary')
        rows.append({
            'job_id':     int(card['data-job-id']),
            'title':      text_at('.job-title a').title(),
            'company':    text_at('.company'),
            'location':   text_at('.location'),
            'job_type':   text_at('.job-type'),
            'salary_min': float(salary.split('-')[0].replace('$', '').replace(',', '').strip()) if salary else None,
            'tags':       [t.get_text(strip=True) for t in card.select('.tag')],
            'posted':     card.select_one('.posted time')['datetime'],
        })
    return rows


jobs = pd.DataFrame(parse_jobs(PAGES / 'job_board.html')
                    + parse_jobs(PAGES / 'job_board_page2.html'))

jobs['posted']    = pd.to_datetime(jobs['posted'])
jobs['is_remote'] = jobs['location'].str.lower().eq('remote')

print(jobs.shape)
print(jobs.dtypes)
jobs

---

## 4.1 The Four Formats

| | CSV | JSON | Excel | Parquet |
|---|---|---|---|---|
| **Layout** | row-based text | nested text | row-based binary | **columnar** binary |
| **Stores types?** | no — everything is text | partly (string/number/bool/null) | partly | **yes, exactly** |
| **Nested data?** | no | **yes** | no | yes (structs, lists) |
| **Compressed?** | no | no | zipped internally | **yes, by column** |
| **Human-readable?** | yes | yes | only in Excel | no |
| **Append a row?** | trivial | awkward | awkward | no (write new files) |
| **Read one column only?** | no — reads everything | no | no | **yes** |
| **Typical size** | 1× | 1.5–3× | 0.5× | **0.1–0.3×** |
| **Tooling everywhere?** | **yes** | yes | yes | needs a library |

### How to choose

| Situation | Format |
|-----------|--------|
| Raw API or scraper output, nested, keep it as it arrived | **JSON** (or JSONL) |
| A log or stream you append to as you go | **JSONL** |
| Handing data to a non-technical colleague | **CSV**, or Excel if they insist |
| Someone needs multiple tabs, or formatting | **Excel** |
| Intermediate storage between pipeline stages | **Parquet** |
| A data lake / warehouse staging layer | **Parquet**, partitioned |
| A table under 1,000 rows that a human will open | **CSV** |
| Analytics over millions of rows | **Parquet** |

### The one line worth memorising

> **CSV to leave the pipeline, Parquet to stay inside it.**

Humans and foreign systems get CSV. Everything that passes from one stage of your own pipeline to the next should be Parquet: it is smaller, faster, and — the part that actually saves you — it still has the types you gave it.

---

## 4.2 CSV — Simple, Universal, Lossy

CSV is a text file with one row per line. There is no type information anywhere in it. `2026-03-02` is a string, `1001` is a string, `True` is a string. When pandas reads a CSV it **guesses** types by sampling the values — and a guess is something that can be wrong.

Watch what a round-trip costs you.

In [ ]:
csv_path = OUT / 'jobs.csv'
jobs.to_csv(csv_path, index=False)      # index=False: do not write the row numbers

reloaded = pd.read_csv(csv_path)

comparison = pd.DataFrame({
    'before': jobs.dtypes.astype(str),
    'after':  reloaded.dtypes.astype(str),
})
comparison['lost'] = comparison['before'] != comparison['after']
comparison

In [ ]:
# Two things broke. Look at the actual values:
print('posted — was a timestamp, is now a string:')
print('  ', repr(reloaded.loc[0, 'posted']), type(reloaded.loc[0, 'posted']).__name__)
print()
print('tags — was a Python list, is now a string that LOOKS like a list:')
print('  ', repr(reloaded.loc[0, 'tags']), type(reloaded.loc[0, 'tags']).__name__)
print()
print('...so this does something useless:')
print('  len(tags) =', len(reloaded.loc[0, 'tags']), '(characters, not tags)')

### Fixing it on the way in

You cannot make CSV store types, but you can tell pandas what to expect instead of letting it guess. Declaring types on read is a habit worth having in every pipeline — it turns a silent wrong guess into a loud error.

| Argument | What it does |
|----------|--------------|
| `parse_dates=['posted']` | parse these columns as datetimes |
| `dtype={'job_id': 'int64'}` | force a column's type instead of inferring it |
| `usecols=[...]` | read only the columns you need |
| `na_values=['N/A', 'none', '-']` | extra strings that mean "missing" |
| `keep_default_na=False` | stop treating `NA`, `null`, `nan` as missing (matters when "NA" is Namibia) |
| `nrows=1000` | peek at a huge file |
| `chunksize=50_000` | iterate over a file too big for memory |
| `thousands=','` | parse `1,234` as a number |
| `sep=';'` / `sep='\t'` | a file that is not comma-separated |
| `encoding='latin-1'` | when UTF-8 raises `UnicodeDecodeError` |

Nested columns — like `tags` — have no CSV representation at all. Either serialise them yourself (`json.dumps`), or do not use CSV.

In [ ]:
# Declare what you expect, and parse the list back out of its string form
fixed = pd.read_csv(
    csv_path,
    parse_dates=['posted'],
    dtype={'job_id': 'int64', 'company': 'string'},
)

# tags was written as "['python', 'sql']" — turn it back into a list
import ast
fixed['tags'] = fixed['tags'].apply(ast.literal_eval)

print(fixed.dtypes)
print()
print('posted:', fixed.loc[0, 'posted'], '|', type(fixed.loc[0, 'posted']).__name__)
print('tags:  ', fixed.loc[0, 'tags'], '|', len(fixed.loc[0, 'tags']), 'tags')

# Better: store nested fields as real JSON, which is unambiguous
jobs_json_tags = jobs.copy()
jobs_json_tags['tags'] = jobs_json_tags['tags'].apply(json.dumps)
jobs_json_tags.to_csv(OUT / 'jobs_jsontags.csv', index=False)
print()
print('tags column as JSON:', pd.read_csv(OUT / "jobs_jsontags.csv").loc[0, "tags"])

In [ ]:
# Not every "CSV" is comma-separated. European exports often use ; and , for decimals.
messy = '''job_id;title;salary_min;notes
1001;Data Engineer;45000,50;"Lagos, Nigeria"
1002;Analytics Engineer;N/A;"Remote; flexible hours"
'''
(OUT / 'semicolon.csv').write_text(messy, encoding='utf-8')

# Default read: one useless column
print('read with defaults:')
print(pd.read_csv(OUT / 'semicolon.csv').columns.tolist())

# Told what it really is:
euro = pd.read_csv(
    OUT / 'semicolon.csv',
    sep=';',
    decimal=',',               # 45000,50 means 45000.50
    na_values=['N/A'],
)
print()
print(euro)
print()
print(euro.dtypes)

### Encoding: `utf-8` vs `utf-8-sig`

Write a CSV containing non-ASCII characters — `£`, `é`, `naïve`, an African place name with a diacritic — open it in Excel on Windows, and the characters are mangled. Excel does not assume UTF-8; it wants a **BOM** (byte order mark) at the start of the file to recognise it.

```python
df.to_csv('for_excel.csv', index=False, encoding='utf-8-sig')   # Excel-safe
df.to_csv('for_pipeline.csv', index=False)                      # plain utf-8
```

`utf-8-sig` writes those three extra bytes. Any sane reader ignores them, and Excel gets it right. Use it for any CSV a colleague will open in Excel.

Reading the other direction: if `pd.read_csv()` raises `UnicodeDecodeError`, the file is not UTF-8. Try `encoding='utf-8-sig'`, then `'latin-1'` (which never fails but may give you mojibake — see the `Â£` bug in Lesson 2).

In [ ]:
# A file with non-ASCII content, written both ways
sample = pd.DataFrame({
    'city':  ['Lagos', 'Abidjan', 'Kinshasa'],
    'label': ['naïve estimate', 'coût estimé', '£1,200 per month'],
})

plain = OUT / 'utf8.csv'
withbom = OUT / 'utf8_sig.csv'
sample.to_csv(plain, index=False, encoding='utf-8')
sample.to_csv(withbom, index=False, encoding='utf-8-sig')

print('plain utf-8 first bytes: ', plain.read_bytes()[:8])
print('utf-8-sig first bytes:   ', withbom.read_bytes()[:8], '  <- the BOM: ef bb bf')
print()
# Both read back correctly in pandas; only the second opens cleanly in Excel
print(pd.read_csv(withbom, encoding='utf-8-sig')['label'].tolist())

---

## 4.3 JSON — Keep the Shape It Arrived In

JSON is the right place to put raw API responses and raw scraper output, because it can hold the nesting those sources actually have. The discipline worth adopting:

> **Save the raw response unchanged, then transform.** If your cleaning logic turns out to be wrong, you re-run it against the saved raw file instead of re-fetching 10,000 pages.

Reading JSON in pandas:

| Call | Use |
|------|-----|
| `json.load(f)` | plain Python dicts/lists — always works, you control everything |
| `pd.read_json(path)` | a file that is already a flat array of objects |
| `pd.json_normalize(data)` | **flatten nested structures into columns** |
| `pd.read_json(path, lines=True)` | JSON **Lines** — one object per line |

`json_normalize` is the one that earns its keep.

In [ ]:
raw = json.loads((DATA / 'sample_jobs.json').read_text(encoding='utf-8'))

print('top-level keys:', list(raw.keys()))
print()
print('one job, nested two levels deep:')
print(json.dumps(raw['jobs'][0], indent=2))

In [ ]:
# Naive: pandas keeps the nested dicts as objects — unusable in a query
naive = pd.DataFrame(raw['jobs'])
print(naive[['job_id', 'company', 'location', 'salary']].head(3))
print()
print('company column holds dicts:', type(naive.loc[0, 'company']).__name__)

In [ ]:
# json_normalize flattens nested keys into dotted column names
flat = pd.json_normalize(raw['jobs'])
print(flat.columns.tolist())
print()
flat.head(3)

In [ ]:
# Give the columns usable names and the right types
flat = flat.rename(columns={
    'company.name':     'company',
    'company.industry': 'industry',
    'location.city':    'city',
    'location.country': 'country',
    'location.remote':  'is_remote',
    'salary.min':       'salary_min',
    'salary.max':       'salary_max',
    'salary.currency':  'currency',
})
flat['posted'] = pd.to_datetime(flat['posted'])

print(flat.dtypes)
flat[['job_id', 'title', 'company', 'industry', 'city', 'is_remote', 'salary_min']]

In [ ]:
# record_path + meta: explode a nested LIST into one row per item,
# carrying parent fields along. Here: one row per (job, tag).
job_tags = pd.json_normalize(
    raw['jobs'],
    record_path='tags',                      # the list to explode
    meta=['job_id', 'title', ['company', 'name']],   # parent fields to keep
)
job_tags.columns = ['tag', 'job_id', 'title', 'company']

print(f'{len(raw["jobs"])} jobs -> {len(job_tags)} job-tag rows\n')
print(job_tags.head(8))
print()
print('Most requested skills:')
print(job_tags['tag'].value_counts())

In [ ]:
# The same idea without json_normalize: DataFrame.explode()
exploded = jobs[['job_id', 'title', 'tags']].explode('tags').rename(columns={'tags': 'tag'})
print(f'{len(jobs)} jobs -> {len(exploded)} rows')
print(exploded.head(6))

# This one-row-per-value shape is what you load into a relational table:
# job_postings (job_id, title, ...) and job_tags (job_id, tag) — Week 2's normalisation,
# applied to scraped data.

### JSON Lines — the format pipelines actually use

A JSON array has to be complete to be valid: you cannot append to it without rewriting the closing bracket, and you cannot read it without loading the whole thing.

**JSON Lines** (`.jsonl`, also called NDJSON) fixes both. One JSON object per line:

```
{"job_id": 1001, "title": "Data Engineer"}
{"job_id": 1002, "title": "Analytics Engineer"}
```

- **Appendable** — a scraper writes one line per record as it goes, and a crash costs you the last line instead of the whole file
- **Streamable** — process a 50 GB file one line at a time, in constant memory
- **Splittable** — any chunk of lines is valid on its own, which is why Spark, BigQuery and friends prefer it

```python
df.to_json(path, orient='records', lines=True)
pd.read_json(path, lines=True)
```

In [ ]:
jsonl_path = OUT / 'jobs.jsonl'
jobs.to_json(jsonl_path, orient='records', lines=True)

print('First two lines of the file:')
for line in jsonl_path.read_text(encoding='utf-8').splitlines()[:2]:
    print(' ', line[:110], '...')

print()
back = pd.read_json(jsonl_path, lines=True)
print('round-tripped:', back.shape)
print('tags survived as a real list:', back.loc[0, 'tags'])
print('posted came back as:', back['posted'].dtype)     # JSON stores it as epoch ms

In [ ]:
# A scraper appending records as it finds them — the real reason for JSONL
stream_path = OUT / 'scrape_stream.jsonl'
stream_path.unlink(missing_ok=True)

with stream_path.open('a', encoding='utf-8') as f:
    for row in jobs.head(3).to_dict(orient='records'):
        row['posted'] = row['posted'].strftime('%Y-%m-%d')      # make it JSON-safe
        f.write(json.dumps(row) + '\n')
        # ...a crash here still leaves you the rows written so far

print(stream_path.read_text(encoding='utf-8'))

# Streaming read: constant memory regardless of file size
with stream_path.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        print(f'{i}: {record["job_id"]}  {record["title"]}')

---

## 4.4 Excel — the Format You Cannot Avoid

Nobody designs a pipeline around Excel. Everybody ends up reading one, because that is what the finance team sends.

```python
pd.read_excel(path, sheet_name='Q1', skiprows=3, usecols='B:F')
df.to_excel(path, index=False, sheet_name='jobs')
```

| Argument | What it does |
|----------|--------------|
| `sheet_name='Q1'` | one sheet by name |
| `sheet_name=0` | the first sheet |
| `sheet_name=None` | **every** sheet, as a `{name: DataFrame}` dict |
| `skiprows=3` | skip a title block above the real header |
| `usecols='B:F'` | a column range, Excel-style |
| `header=None` | the sheet has no header row |

### What to know going in

- **It needs a library.** `pip install openpyxl` for `.xlsx`. (`.xls` is a different, obsolete format needing `xlrd`.)
- **It is slow.** Writing 100,000 rows to `.xlsx` takes seconds; Parquet takes milliseconds.
- **It has a hard limit** of 1,048,576 rows per sheet. Real pipelines hit this.
- **Real workbooks are messy** — merged cells, a title above the header, footnotes under the data, numbers stored as text, several tables on one sheet. Expect `skiprows` and `usecols`.
- **pandas ignores formatting, formulas and charts.** A formula cell comes in as its last cached *value*. To keep formatting, use `openpyxl` directly.
- **Never use Excel as a pipeline hand-off between your own stages.** Use it only at the boundary where a human is involved.

In [ ]:
# Write a multi-sheet workbook with ExcelWriter
xlsx_path = OUT / 'job_report.xlsx'

by_company = (jobs.groupby('company')
                  .agg(postings=('job_id', 'count'), avg_salary=('salary_min', 'mean'))
                  .reset_index()
                  .sort_values('postings', ascending=False))

by_type = jobs['job_type'].value_counts().rename_axis('job_type').reset_index(name='postings')

# tags is a list column — Excel cannot hold a list, so flatten it to a string
for_excel = jobs.copy()
for_excel['tags'] = for_excel['tags'].apply(', '.join)

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    for_excel.to_excel(writer, sheet_name='postings',  index=False)
    by_company.to_excel(writer, sheet_name='by_company', index=False)
    by_type.to_excel(writer,    sheet_name='by_type',    index=False)

print(f'wrote {xlsx_path}  ({xlsx_path.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Read every sheet at once: a dict of DataFrames
sheets = pd.read_excel(xlsx_path, sheet_name=None)

print('sheets in the workbook:')
for name, frame in sheets.items():
    print(f'  {name:<12} {frame.shape}')

print()
print(sheets['by_company'])

In [ ]:
# A realistic messy sheet: two title rows above the header, a footnote below the data
messy_path = OUT / 'messy_report.xlsx'

with pd.ExcelWriter(messy_path, engine='openpyxl') as writer:
    # Rows 0-1 are a title block, row 2 is the real header
    pd.DataFrame([['AfriTech Jobs — Internal Report'], ['Generated 2026-03-07']]).to_excel(
        writer, sheet_name='report', index=False, header=False, startrow=0)
    by_company.to_excel(writer, sheet_name='report', index=False, startrow=2)
    pd.DataFrame([['* salaries are minimums, USD']]).to_excel(
        writer, sheet_name='report', index=False, header=False,
        startrow=3 + len(by_company) + 1)

# Reading it naively gives you nonsense column names and a footnote row of data
naive_read = pd.read_excel(messy_path, sheet_name='report')
print('naive read:')
print(naive_read.head(3))
print(f'  columns: {naive_read.columns.tolist()}')

# Reading it correctly: skip the title block, take only the real rows
clean_read = pd.read_excel(messy_path, sheet_name='report',
                           skiprows=2, nrows=len(by_company))
print()
print('skiprows=2, nrows=', len(by_company), ':')
print(clean_read)

---

## 4.5 Parquet — the One You Should Default To

Parquet stores data **by column** rather than by row, in a binary file with a schema in its footer.

```
CSV (row-based):        1001,Data Engineer,Paystack,45000
                        1002,Analytics Engineer,Flutterwave,

Parquet (columnar):     job_id:  [1001, 1002, 1003, ...]        <- all ints together
                        title:   ["Data Engineer", ...]         <- all strings together
                        salary:  [45000, null, 18000, ...]
```

Four consequences, all good:

1. **Types are stored, not guessed.** A timestamp comes back a timestamp; an int64 comes back int64; a list column comes back a list. No `parse_dates`, no surprises.
2. **It compresses hard.** A column of values of one type, sorted together, compresses far better than mixed-type rows. Expect 5–10× smaller than the same CSV.
3. **You can read one column.** `pd.read_parquet(path, columns=['salary_min'])` reads only that column's bytes off disk. On a 200-column table that is a transformative difference, and it is why every warehouse engine — BigQuery, Snowflake, Spark, DuckDB, Athena — uses a columnar format underneath.
4. **It carries a schema.** Readers can validate before loading, and column types cannot silently drift between runs.

The trade-offs: it is not human-readable, you cannot append to a file (you write new files instead), and reading it needs a library (`pyarrow`).

In [ ]:
parquet_path = OUT / 'jobs.parquet'
jobs.to_parquet(parquet_path, index=False)        # default compression: snappy

pq = pd.read_parquet(parquet_path)

print(pd.DataFrame({
    'original':    jobs.dtypes.astype(str),
    'from_csv':    pd.read_csv(csv_path).dtypes.astype(str),
    'from_parquet': pq.dtypes.astype(str),
}))

print()
print('posted:', repr(pq.loc[0, 'posted']))
print('tags:  ', pq.loc[0, 'tags'], f'({type(pq.loc[0, "tags"]).__name__}, {len(pq.loc[0, "tags"])} items)')
print()
print('Nothing was lost. No parse_dates, no literal_eval, no guessing.')

In [ ]:
# Read only the columns you need — Parquet reads only those bytes off disk
two_cols = pd.read_parquet(parquet_path, columns=['company', 'salary_min'])
print(two_cols)

# Inspect the schema without reading any data at all
import pyarrow.parquet as pq_mod
meta = pq_mod.read_metadata(parquet_path)
print()
print(f'rows: {meta.num_rows}   columns: {meta.num_columns}   row groups: {meta.num_row_groups}')
print()
print(pq_mod.read_schema(parquet_path))

---

## 4.6 Measuring It

Ten rows prove nothing. Let us write the same 20,000-row table in every format and measure size and read time.

The numbers below are from one machine and will differ on yours. The **ratios** are what matter, and they hold everywhere.

In [ ]:
# Build a 20,000-row version of the same data
big = pd.concat([jobs] * 2000, ignore_index=True)
big['job_id'] = range(1, len(big) + 1)
big['tags_str'] = big['tags'].apply(', '.join)      # a CSV/Excel-safe copy

print(f'{len(big):,} rows x {len(big.columns)} columns')
big.dtypes

In [ ]:
def timed(fn):
    '''Run fn() and return (result, seconds).'''
    t0 = time.perf_counter()
    result = fn()
    return result, time.perf_counter() - t0


flat_big = big.drop(columns=['tags'])     # CSV and Excel cannot hold a list column
results = []

# --- CSV ---
p = OUT / 'big.csv'
_, w = timed(lambda: flat_big.to_csv(p, index=False))
_, r = timed(lambda: pd.read_csv(p))
results.append(('CSV', p.stat().st_size, w, r))

# --- CSV, gzipped ---
p = OUT / 'big.csv.gz'
_, w = timed(lambda: flat_big.to_csv(p, index=False, compression='gzip'))
_, r = timed(lambda: pd.read_csv(p))
results.append(('CSV + gzip', p.stat().st_size, w, r))

# --- JSON (array) ---
p = OUT / 'big.json'
_, w = timed(lambda: flat_big.to_json(p, orient='records'))
_, r = timed(lambda: pd.read_json(p))
results.append(('JSON', p.stat().st_size, w, r))

# --- JSON Lines ---
p = OUT / 'big.jsonl'
_, w = timed(lambda: flat_big.to_json(p, orient='records', lines=True))
_, r = timed(lambda: pd.read_json(p, lines=True))
results.append(('JSONL', p.stat().st_size, w, r))

# --- Excel ---
p = OUT / 'big.xlsx'
_, w = timed(lambda: flat_big.to_excel(p, index=False))
_, r = timed(lambda: pd.read_excel(p))
results.append(('Excel (.xlsx)', p.stat().st_size, w, r))

# --- Parquet (snappy, the default) ---
p = OUT / 'big.parquet'
_, w = timed(lambda: big.to_parquet(p, index=False))
_, r = timed(lambda: pd.read_parquet(p))
results.append(('Parquet snappy', p.stat().st_size, w, r))

# --- Parquet (gzip: smaller, slower) ---
p = OUT / 'big_gzip.parquet'
_, w = timed(lambda: big.to_parquet(p, index=False, compression='gzip'))
_, r = timed(lambda: pd.read_parquet(p))
results.append(('Parquet gzip', p.stat().st_size, w, r))

# --- Parquet, one column only ---
p = OUT / 'big.parquet'
_, r1 = timed(lambda: pd.read_parquet(p, columns=['salary_min']))
results.append(('Parquet 1 column', p.stat().st_size, float('nan'), r1))

bench = pd.DataFrame(results, columns=['format', 'bytes', 'write_s', 'read_s'])
bench['size_mb']  = (bench['bytes'] / 1_048_576).round(2)
bench['vs_csv']   = (bench['bytes'] / bench.loc[0, 'bytes']).round(2)
bench['write_s']  = bench['write_s'].round(3)
bench['read_s']   = bench['read_s'].round(3)

bench[['format', 'size_mb', 'vs_csv', 'write_s', 'read_s']]

### Reading that table

The pattern you should expect, every time:

- **Parquet is several times smaller than CSV** — same data, same fidelity, a fraction of the disk and a fraction of the bytes over the network. On cloud storage you pay for both.
- **JSON is the biggest.** Every row repeats every key name. Useful for nesting, wasteful for tables.
- **Excel is by far the slowest to write.** It is a zipped XML document; every cell is a tag.
- **Reading one Parquet column is dramatically faster than reading the file**, because the other columns are never touched. CSV cannot do this at any price — it has to parse every line to find the commas.
- **gzip beats snappy on size, snappy beats gzip on speed.** Snappy (the default) is the right call for pipeline intermediates; gzip or zstd for cold archives.

### Two honest caveats

**Our benchmark data is unusually compressible.** We built it by repeating ten rows two thousand times, so `gzip` has an easy job — you may well see **gzipped CSV come out smaller than snappy Parquet** in the table above. That is a property of the test data, not a general result; on real data with diverse values Parquet normally wins outright, and switching Parquet to `gzip` or `zstd` puts it back in front.

More importantly, size is not the only axis. A `.csv.gz` still has to be fully decompressed and fully parsed on every read, still has no types, and still cannot read one column — compare the read times and the `Parquet 1 column` row. Compression narrows the size gap; it does nothing about the rest.

**On small data, none of this matters.** For a 100-row table, CSV's simplicity wins and the timings are noise. Parquet's advantages are about scale — and about **types**, which matter at every size.

---

## 4.7 Partitioned Datasets

Once a table is large, you stop writing one file and write a **directory** instead, split by the column you filter on most:

```
data/jobs/
├── country=Kenya/
│   └── part-0.parquet
├── country=Nigeria/
│   └── part-0.parquet
└── country=Rwanda/
    └── part-0.parquet
```

That layout is called **Hive partitioning**, and every warehouse tool understands it. The payoff is **partition pruning**: a query filtered to Kenya opens one directory and never touches the others. This is how a terabyte-scale table answers a question in seconds, and it is exactly what you will meet again in Week 6 (warehousing), Week 8 (cloud storage) and Week 9 (dbt).

Partition on a column that is **low-cardinality** and **frequently filtered** — date, country, source. Never partition on something like `job_id`: 20,000 directories of one row each is slower than no partitioning at all.

In [ ]:
import shutil

# Partition by country: pandas writes the directory tree for you
part_dir = OUT / 'jobs_partitioned'
shutil.rmtree(part_dir, ignore_errors=True)        # start clean on a re-run

partitioned = big.copy()
partitioned['country'] = partitioned['location'].str.split(',').str[-1].str.strip()

partitioned.to_parquet(part_dir, partition_cols=['country'], index=False)

print('Directory layout:')
for d in sorted(p for p in part_dir.iterdir() if p.is_dir()):
    files = list(d.glob('*.parquet'))
    size  = sum(f.stat().st_size for f in files) / 1024
    print(f'  {d.name}/   {len(files)} file(s), {size:,.0f} KB')

total_unsorted = sum(f.stat().st_size for f in part_dir.rglob('*.parquet'))
print(f'\ntotal on disk: {total_unsorted / 1_048_576:.2f} MB')

### Wait — that is bigger than the single file

The same 20,000 rows were 0.12 MB as one Parquet file. Partitioned, they take **tens of megabytes**. Partitioning made it 100× worse.

Here is why. Parquet stores data in **row groups** — blocks of rows that get compressed together, each with its own footer of statistics. Compression works because similar values sit next to each other in a block. When the writer splits your rows by partition key, it keeps the order you gave it, and our rows arrive **interleaved**: Nigeria, Remote, Kenya, Remote, Ghana, Kenya, ... Every single row is its own contiguous run, so the writer emits a row group per row. Twenty thousand row groups of one row each: all footer, no compression.

This is the **small files problem**, and it is one of the most common performance faults in real data lakes. The fix is one line:

> **Sort by the partition column before writing a partitioned dataset.**

Then each partition is one long contiguous run, the writer emits a few large row groups, and compression has something to work with.

In [ ]:
# The same write, sorted by the partition key first
sorted_dir = OUT / 'jobs_partitioned_sorted'
shutil.rmtree(sorted_dir, ignore_errors=True)

(partitioned
 .sort_values('country')                 # <-- the entire fix
 .to_parquet(sorted_dir, partition_cols=['country'], index=False))

total_sorted = sum(f.stat().st_size for f in sorted_dir.rglob('*.parquet'))

def row_groups(directory):
    return sum(pq_mod.read_metadata(f).num_row_groups
               for f in directory.rglob('*.parquet'))

print(f'{"":10} {"size":>12}  {"row groups":>11}')
print(f'{"unsorted":10} {total_unsorted / 1_048_576:>9.2f} MB  {row_groups(part_dir):>11,}')
print(f'{"sorted":10} {total_sorted / 1_048_576:>9.2f} MB  {row_groups(sorted_dir):>11,}')
print(f'\n{total_unsorted / total_sorted:.0f}x smaller, from one sort_values().')

In [ ]:
# Read the whole dataset: country comes back as a column, reconstructed from the path
whole, t_whole = timed(lambda: pd.read_parquet(sorted_dir))
print(f'whole dataset:     {len(whole):,} rows in {t_whole:.3f}s')
print(whole['country'].value_counts().to_string())

# Read one partition: only those files are opened
kenya, t_kenya = timed(lambda: pd.read_parquet(sorted_dir, filters=[('country', '==', 'Kenya')]))
print(f'\nKenya only:        {len(kenya):,} rows in {t_kenya:.3f}s')
print(f'speed-up:          {t_whole / max(t_kenya, 1e-9):.1f}x')

# filters works on non-partition columns too (row-group statistics do the pruning)
rich, _ = timed(lambda: pd.read_parquet(OUT / 'big.parquet',
                                        filters=[('salary_min', '>=', 50000)]))
print(f'\nsalary >= 50,000:  {len(rich):,} rows')

In [ ]:
# Finally: save this week's dataset the way a real pipeline would.
# Parquet for the pipeline, CSV for the humans.
jobs.to_parquet(CLEAN / 'jobs.parquet', index=False)

human_copy = jobs.copy()
human_copy['tags'] = human_copy['tags'].apply(', '.join)
human_copy.to_csv(CLEAN / 'jobs.csv', index=False, encoding='utf-8-sig')

print(f'{CLEAN / "jobs.parquet"}  {(CLEAN / "jobs.parquet").stat().st_size:,} bytes  (pipeline)')
print(f'{CLEAN / "jobs.csv"}     {(CLEAN / "jobs.csv").stat().st_size:,} bytes  (humans)')

---

## Key Takeaways

1. **CSV to leave the pipeline, Parquet to stay inside it.** Humans and foreign systems get CSV; everything passing between your own stages should be Parquet.
2. CSV stores **no types**. Every round-trip loses your dates, your booleans-as-strings and any nested column, and pandas guesses the rest. Guesses are sometimes wrong.
3. Declare types when you read a CSV: `parse_dates=`, `dtype=`, `na_values=`, `usecols=`. It turns a silent wrong guess into a loud error.
4. Not every `.csv` is comma-separated: `sep=';'`, `decimal=','`, `thousands=','` and `encoding=` exist for a reason.
5. Write CSVs destined for Excel with **`encoding='utf-8-sig'`**, or your non-ASCII characters will be mangled on a colleague's Windows machine.
6. **Save raw API and scraper output as JSON, unchanged**, then transform. Re-running your cleaning against a saved file is free; re-fetching 10,000 pages is not.
7. `pd.json_normalize()` flattens nested JSON into dotted columns; `record_path` + `meta` explodes a nested list into one row per item — exactly the shape a relational table wants.
8. **JSON Lines** (`orient='records', lines=True`) is the format pipelines use: appendable, streamable in constant memory, splittable, and crash-tolerant.
9. **Excel** is a boundary format — fine for a human deliverable, never for a hand-off between your own pipeline stages. It is slow, caps at ~1M rows, and needs `openpyxl`. Expect `skiprows` and `usecols` on real workbooks, and `sheet_name=None` to read them all.
10. **Parquet is columnar, typed and compressed**: several times smaller than CSV, types preserved exactly, and `columns=` reads one column without touching the rest. That is why every warehouse engine is built on it.
11. `snappy` (the default) for speed, `gzip`/`zstd` for archives. Both beat uncompressed CSV on size by a wide margin.
12. **Partition large Parquet datasets** by a low-cardinality column you filter on often (date, country, source). `filters=[...]` then prunes whole directories. Never partition on a high-cardinality id.
13. **Sort by the partition column before a partitioned write.** Interleaved keys make the writer emit one tiny row group per row — the small files problem — and can cost you 100× in size. One `sort_values()` fixes it.

---

**Next:** Lab 5 — put Lessons 1–4 together: scrape a live job board with Playwright, clean it, store it as CSV and Parquet, and load it into PostgreSQL.